# PaddleOCR-VL fine-tuning on Colab (ERNIEKit SFT): NOT RUN, NOT VERIFIED

Prepared for TactileLens; **this notebook has never been run**. It follows the ERNIEKit guide
(https://ernie.readthedocs.io/en/develop/paddleocr_vl_sft.html) as of October 2026.

- Fine-tunes **our own copy** of `PaddlePaddle/PaddleOCR-VL` (0.9B, ~2 GiB download), full fine-tuning
  as in ERNIEKit's example config (`run_ocr_vl_sft_16k.yaml`: bf16, flash attention, recompute).
- Baidu's hosted API **cannot** be fine-tuned. A model trained here would have to run on **our own GPU**;
  the app's hosted engine would not use it.
- Needs a GPU with **bf16 and flash attention** (Ampere or newer, compute capability >= 8.0) and, by our
  estimate, **40 GB or more** (A100). A free T4 (16 GB, Turing) does **not** fit: see
  `docs/vl_finetuning_feasibility.md`.
- Upload only our own images (no third-party or textbook pages, no personal data). The `eval` split is
  never given to training.

Input: a zip on Google Drive holding the data folder with the output of `convert_to_sft.py` in `sft/`
(made with `--out <data>/sft`, so the relative image links stay valid).

In [ ]:
# 1. GPU check: stop unless the GPU supports bf16 / flash attention and has >= 39 GB.
import subprocess
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.strip()
print(info)
name, memory_mib, capability = [s.strip() for s in info.splitlines()[0].split(",")]
ALLOW_UNSUPPORTED_GPU = False  # only change this knowingly; the example config will not run on a T4
if (float(capability) < 8.0 or int(memory_mib) < 39 * 1024) and not ALLOW_UNSUPPORTED_GPU:
    raise SystemExit(f"{name}: compute capability {capability}, {int(memory_mib) // 1024} GiB. "
                     "Choose an A100 runtime (Runtime > Change runtime type).")

In [ ]:
# 2. Settings
from google.colab import drive
drive.mount("/content/drive")

DATA_ZIP = "/content/drive/MyDrive/tactilelens_vl/vl_data.zip"   # data folder with sft/ inside
RUNS_ON_DRIVE = "/content/drive/MyDrive/tactilelens_vl/runs"
ERNIE_REF = "develop"        # pin a commit once a run works, for repeatability
EPOCHS = 2
GRAD_ACCUMULATION = 8        # as in the example config (1 GPU)
MAX_SEQ_LEN = 16384          # example config; 8192 is enough for single-formula crops
SAVE_STEPS = 200

In [ ]:
# 3. Unpack, make image links absolute, check every image and the eval leak guard.
import hashlib, json, os, zipfile
from pathlib import Path

DATA = Path("/content/vl_data")
with zipfile.ZipFile(DATA_ZIP) as archive:
    archive.extractall(DATA)
sft = next(p.parent for p in DATA.rglob("train.jsonl"))
print("sft folder:", sft)

def records(split):
    path = sft / f"{split}.jsonl"
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

def meta(split):
    path = sft / f"{split}_meta.jsonl"
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

eval_hashes = {m["sha256"] for m in meta("eval")}
eval_names = {m["filename"] for m in meta("eval")}
TRAIN_FILES = {}
for split in ("train", "val"):
    rows = []
    for rec, m in zip(records(split), meta(split)):
        image = Path(rec["image_info"][0]["image_url"])
        image = image if image.is_absolute() else (sft / image).resolve()
        assert image.exists(), f"missing image {image}"
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        assert digest == m["sha256"], f"{image} changed since convert_to_sft.py"
        assert digest not in eval_hashes and m["filename"] not in eval_names, f"LEAK: {m['filename']} is an eval image"
        rec["image_info"][0]["image_url"] = str(image)
        rows.append(rec)
    out = Path(f"/content/{split}_abs.jsonl")
    out.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    TRAIN_FILES[split] = (str(out), len(rows))
print({k: n for k, (_, n) in TRAIN_FILES.items()}, "| eval held out:", len(eval_hashes))

In [ ]:
# 4. Install PaddlePaddle 3.2 (CUDA 12.6) and ERNIEKit, as in the guide.
# The paddlepaddle-gpu line follows the PaddlePaddle install guide; check it there before running.
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -c "import paddle; paddle.utils.run_check()"
!git clone https://github.com/PaddlePaddle/ERNIE /content/ERNIE && cd /content/ERNIE && git checkout {ERNIE_REF}
!cd /content/ERNIE && python -m pip install -r requirements/gpu/requirements.txt && python -m pip install -e .
!python -m pip install tensorboard opencv-python-headless numpy==1.26.4

In [ ]:
# 5. Download the base model (~2 GiB).
!huggingface-cli download PaddlePaddle/PaddleOCR-VL --local-dir /content/PaddleOCR-VL

In [ ]:
# 6. Train. max_steps ~ (D x E) / (G x B x A) and warmup ~1% of it, as the guide says.
import math, time
train_file, n_train = TRAIN_FILES["train"]
val_file, n_val = TRAIN_FILES["val"]
MAX_STEPS = max(1, math.ceil(n_train * EPOCHS / (1 * 1 * GRAD_ACCUMULATION)))
WARMUP = max(1, MAX_STEPS // 100)
RUN = f"{RUNS_ON_DRIVE}/{time.strftime('%Y%m%d_%H%M%S')}"
OUT = "/content/PaddleOCR-VL-SFT-TactileLens"
print(f"{n_train} train records, {n_val} val; max_steps {MAX_STEPS}, warmup {WARMUP}; output {OUT}")

!cd /content/ERNIE && CUDA_VISIBLE_DEVICES=0 erniekit train examples/configs/PaddleOCR-VL/sft/run_ocr_vl_sft_16k.yaml \
    model_name_or_path=/content/PaddleOCR-VL \
    train_dataset_path={train_file} \
    eval_dataset_path={val_file} \
    num_train_epochs={EPOCHS} max_steps={MAX_STEPS} warmup_steps={WARMUP} \
    gradient_accumulation_steps={GRAD_ACCUMULATION} max_seq_len={MAX_SEQ_LEN} save_steps={SAVE_STEPS} \
    output_dir={OUT} logging_dir={OUT}/tensorboard_logs

In [ ]:
# 7. Make the folder loadable by PaddleOCR (as the guide says) and copy it to Drive without checkpoints.
!cp /content/PaddleOCR-VL/chat_template.jinja {OUT}/
!cp /content/PaddleOCR-VL/inference.yml {OUT}/
!mkdir -p "{RUN}" && rsync -a --exclude 'checkpoint-*' {OUT}/ "{RUN}/model/"
!cp /content/train_abs.jsonl /content/val_abs.jsonl "{RUN}/"
print("Saved to", RUN)

## 8. Evaluate on the held-out set (on our PC)

Download `model/` from the run folder, then from `tools/training_vl`:

```
python evaluate_vl.py --eval <data>\sft --engine finetuned --model-dir <run>\model ^
    --vl-model-name PaddleOCR-VL-0.9B --pipeline-version v1
```

Compare with the local PaddleOCR-VL-1.6 (`--engine local`) and the hosted API results (run by the
owner). A fair baseline for this fine-tune is the **un-tuned** PaddleOCR-VL-0.9B (same flags, the
downloaded base model as `--model-dir`); downloading it to the PC is a separate decision. Report only
measured numbers.